# Milestone 1 - verify the WFLW 98→24 mapping

Attach the WFLW dataset (`mrriandmstique/wflw-wider-facial-landmarks-in-the-wild`) to this notebook, then run all cells.

The notebook is deliberately thin: it clones the repo and runs the scripts, so every result is reproducible from a commit. If the repo is private, add a GitHub token to the clone URL via Kaggle Secrets.

**Checklist to confirm on the overlays before anything trains:**
* pupil points **12, 13** dead centre in each eye (crosshair in the insets) - not offset, not on an eyelid, not swapped left/right
* eyelid points **0–5 / 6–11** trace the lids: corner, two upper, corner, two lower - consistent order in every face
* mouth points **14–17** on the corners and outer-lip midpoints, not the lip interior
* nose tip **18** and chin **19** on the vertical facial axis
* contour points **20–23** symmetric left/right at roughly equal height

If the dataset mount path differs from the config default, edit `dataset.root` in `configs/layer1_base.yaml` - the loader prints what it found when the path is wrong.

In [ ]:
!rm -rf dms-layer1
!git clone -b claude/facial-landmark-perception-q80yhn https://github.com/keerthanpragnay1728-prog/dms-layer1.git
%cd dms-layer1
# provenance: confirm the commit this run uses BEFORE trusting any number
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
# unit tests: schema validation, flip permutation, annotation parsing
!python tests/run_tests.py

In [ ]:
# statistical verification of the assumed 98-point index layout
# against the real annotation file (must be all PASS)
!python scripts/verify_layout.py --config configs/layer1_base.yaml --split test

In [ ]:
# magnitude diagnostics for the pose-sensitive layout checks: which contour
# point is actually lowest and by how much, failure rate vs estimated yaw,
# the nose-midline tolerance, and the four checks on the no-flag subset
!python scripts/diagnose_layout_failures.py --config configs/layer1_base.yaml --split test
from IPython.display import Image, display
display(Image(filename='/kaggle/working/m1_diagnostics/failure_rate_vs_yaw.png'))

In [ ]:
# labelled verification overlays -> /kaggle/working/m1_overlays
!python scripts/visualize_mapping.py --config configs/layer1_base.yaml --split test

In [ ]:
# display every overlay inline for the eyeball check
from pathlib import Path
from IPython.display import Image, display

out = Path('/kaggle/working/m1_overlays')
for png in sorted(out.glob('*.png')):
    if png.name != 'contact_sheet.png':
        print(png.name)
        display(Image(filename=str(png)))